# Lab 10: A Grande Integração - Dashboard Agentic Estruturado (JSON Schema + UI)

**Disciplina:** Augmented Analytics & AI-Driven Insights
**Missão:** Integrar o "Cérebro" refinado no AI Studio (Lab 08) à interface Streamlit (Lab 09).

---
### 🚀 1. Instalação de Dependências

In [ ]:
!pip install crewai streamlit pyngrok pandas -q

### 📝 2. Desenvolvimento do Dashboard Integrado (app.py)
Nesta célula, você executará a aplicação completa integrando o Agente inteligente com extração segura de JSON e interface interativa.

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import os
import json
import re
from crewai import Agent, Task, Crew, Process

# -----------------------------------------------------------------
# 1. CONFIGURAÇÃO DA UI
# -----------------------------------------------------------------
st.set_page_config(page_title="BI Pro Dashboard", layout="wide")
st.title("🏢 Agentic Analytics: Insight-as-a-Service")

with st.sidebar:
    st.header("⚙️ Painel de Controle")
    api_key = st.text_input("Gemini API Key:", type="password")
    st.divider()
    st.info("MBA BI & Analytics - FIAP")

# -----------------------------------------------------------------
# 2. LÓGICA DE INTEGRAÇÃO (A CIRURGIA)
# -----------------------------------------------------------------
uploaded_file = st.file_uploader("Suba o dataset para análise estratégica (CSV):", type="csv")

if uploaded_file and api_key:
    df = pd.read_csv(uploaded_file)
    st.dataframe(df.head(10), use_container_width=True)
    
    if st.button("🚀 Gerar Análise Estruturada"):
        os.environ["GEMINI_API_KEY"] = api_key
        
        with st.spinner("🧠 Squad processando dados via JSON Schema... Aguarde."):
            
            # PASSO 1: System Instructions (Tom Executivo e Foco em ROI)
            instrucoes_sistemicas = "Você é um Consultor Estratégico do MBA em BI & Analytics da FIAP. Seu tom é pragmático, focado em ROI e métricas de negócio. Sempre baseie suas conclusões nos dados fornecidos."
            
            analyst = Agent(
                role='Consultor Estratégico FIAP',
                goal='Gerar insights acionáveis e estruturados',
                backstory=instrucoes_sistemicas,
                llm="google/gemini-3-flash-preview"
            )
            
            # PASSO 2: Amostragem representativa para evitar estouro de tokens
            dataset_contexto = df.head(15).to_csv(index=False)
            
            task = Task(
                description=f'''Analise a amostra de dados abaixo e gere um insight estratégico estruturado:\n\n{dataset_contexto}\n\nRetorne OBRIGATORIAMENTE um objeto JSON com o formato:\n{{\n  "titulo": "string",\n  "metrica_critica": "string",\n  "recomendacao": "string",\n  "impacto_estimado": "Baixo" | "Médio" | "Alto"\n}}\n\nRetorne APENAS o JSON puro, sem blocos explicativos ou markdown adicionais.''',
                expected_output='Um objeto JSON com as chaves: titulo, metrica_critica, recomendacao, impacto_estimado.',
                agent=analyst
            )
            
            crew = Crew(agents=[analyst], tasks=[task], process=Process.sequential)
            raw_result = crew.kickoff()
            
            # -----------------------------------------------------------------
            # 3. EXTRAÇÃO ROBUSTA VIA REGEX E PERSISTÊNCIA
            # -----------------------------------------------------------------
            try:
                raw_text = str(raw_result.raw if hasattr(raw_result, 'raw') else raw_result)
                json_match = re.search(r'\{.*\}', raw_text, re.DOTALL)
                
                if json_match:
                    st.session_state["analise_data"] = json.loads(json_match.group(0))
                    st.session_state["raw_output"] = raw_text
                else:
                    raise ValueError("JSON não identificado no retorno do agente.")
                    
            except Exception as e:
                st.error(f"❌ Erro ao processar JSON: {e}")
                st.write("**Retorno Bruto:**", raw_result)

    # -----------------------------------------------------------------
    # 4. EXIBIÇÃO PROFISSIONAL DO DASHBOARD
    # -----------------------------------------------------------------
    if "analise_data" in st.session_state:
        data = st.session_state["analise_data"]
        st.success("✅ Inteligência Gerada com Sucesso!")
        st.divider()
        
        col1, col2 = st.columns(2)
        
        with col1:
            st.metric(label="Título do Insight", value=data.get('titulo', 'N/A'))
            st.write(f"**Métrica Crítica:** {data.get('metrica_critica', 'N/A')}")
        
        with col2:
            st.metric(label="Impacto Estimado", value=data.get('impacto_estimado', 'Médio'))
            st.warning(f"**Recomendação:** {data.get('recomendacao', 'N/A')}")
        
        st.divider()
        st.write("### 🛠️ Raw JSON (Para Auditoria IT)")
        st.json(data)

elif not api_key:
    st.warning("⚠️ Insira sua Gemini API Key no menu lateral para ativar a Squad de Agentes.")

### 🌐 3. Publicação & Túnel
Insira seu Token do Ngrok abaixo e execute a célula para abrir o link público do Dashboard.

In [ ]:
# @title Configuração do Túnel (Ngrok)
NGROK_TOKEN = "" # @param {type:"string"}

from pyngrok import ngrok
import os

if NGROK_TOKEN:
    ngrok.kill()
    ngrok.set_auth_token(NGROK_TOKEN)
    public_url = ngrok.connect(8501).public_url
    print(f"🔗 DASHBOARD PROFISSIONAL ATIVO EM: {public_url}")
    print("--- ")
    !streamlit run app.py --server.port 8501
else:
    print("❌ ERRO: Você precisa inserir o NGROK_TOKEN para gerar o link de acesso.")